# Subsample BCN Noise ML Dataset

This notebook creates a smaller version of `bcn_noise_ml_dataset.csv` and exports it as a new CSV.

Because each row has three noise values (`noise_day`, `noise_evening`, `noise_night`), the most stable approach is to collapse them into one row-level noise score, bin that score into quantile buckets, and then sample equally from each bucket.

If you want a more conservative definition of exposure, you can switch the score from the mean to the maximum of the three noise columns.

## Sampling Strategy

The notebook below uses:

1. `noise_score = mean(noise_day, noise_evening, noise_night)`
2. Quantile buckets with `pd.qcut`
3. A 50% sample from each bucket

This keeps the reduced dataset balanced across low, medium, and high noise segments instead of letting one range dominate the sample.

In [1]:
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
INPUT_PATH = Path('../../data/machine_learning/bcn_noise_ml_dataset.csv')
OUTPUT_PATH = Path('../../data/machine_learning/bcn_noise_ml_dataset_halved.csv')
NOISE_COLUMNS = ['noise_day', 'noise_evening', 'noise_night']
N_BUCKETS = 8
SAMPLE_FRACTION = 0.5

print(f'Input: {INPUT_PATH}')
print(f'Output: {OUTPUT_PATH}')

Input: ..\..\data\machine_learning\bcn_noise_ml_dataset.csv
Output: ..\..\data\machine_learning\bcn_noise_ml_dataset_halved.csv


In [3]:
df = pd.read_csv(INPUT_PATH)

missing_columns = [column for column in NOISE_COLUMNS if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing noise columns: {missing_columns}')

df = df.copy()
df['noise_score'] = df[NOISE_COLUMNS].mean(axis=1)
df['noise_bucket'] = pd.qcut(
    df['noise_score'],
    q=N_BUCKETS,
    labels=False,
    duplicates='drop'
)

bucket_counts = df['noise_bucket'].value_counts(dropna=False).sort_index()
display(bucket_counts.to_frame(name='rows_in_bucket'))
print(f'Original rows: {len(df)}')
print(f"Buckets created: {df['noise_bucket'].nunique(dropna=True)}")

,rows_in_bucket
noise_bucket,
0,3049
1,1410
2,2391
3,1796
4,2462
5,1631
6,2084
7,1949


Original rows: 16772
Buckets created: 8


In [5]:
sampled_with_helpers = (
    df.dropna(subset=['noise_bucket'])
      .groupby('noise_bucket', group_keys=True)
      .apply(lambda bucket: bucket.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE))
      .reset_index(level=0)
      .sort_index()
      .reset_index(drop=True)
)

sampled_bucket_counts = sampled_with_helpers['noise_bucket'].value_counts().sort_index()
display(sampled_bucket_counts.to_frame(name='sampled_rows'))
print(f'Sampled rows: {len(sampled_with_helpers)}')
print(f'Sampled share: {len(sampled_with_helpers) / len(df):.2%}')

sampled_df = sampled_with_helpers.drop(columns=['noise_score', 'noise_bucket'])
if 'fid' in sampled_df.columns:
    sampled_df = sampled_df.sort_values('fid').reset_index(drop=True)
else:
    sampled_df = sampled_df.reset_index(drop=True)

display(sampled_df.head())

,sampled_rows
noise_bucket,
0,1524
1,705
2,1196
3,898
4,1231
5,816
6,1042
7,974


Sampled rows: 8386
Sampled share: 50.00%


,street_id,noise_day,noise_evening,noise_night,road_length,road_category,one_way,distance_to_road,fid,osm_commercial_pct_50m,...,transport_count_50,street_tree_count_10m,park_tree_count_10m,street_tree_count_20m,park_tree_count_20m,total_tree_count_10m,total_tree_count_20m,road_width,openness,edge_betweenness
0,T05360P,55,55,50,91.759758,5,1,0.0,5,0.0,...,1,1,0,11,0,1,11,15.404320,0.454545,0.000943
1,T08863T,50,50,45,86.869284,5,1,0.0,6,0.0,...,1,1,0,2,0,1,2,9.382715,0.200000,0.001085
2,T13009A,55,55,50,147.700343,5,1,0.0,8,0.0,...,4,0,0,1,0,0,1,36.722212,0.710526,0.003190
3,T16868V,45,45,40,153.250633,5,1,0.0,10,0.0,...,1,10,0,11,0,10,11,33.145383,0.675000,0.000834
4,T04571U,60,55,50,94.090297,5,1,0.0,12,0.0,...,1,9,0,13,0,9,13,30.479123,0.590909,0.001827


In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sampled_df.to_csv(OUTPUT_PATH, index=False)

print(f'Exported {len(sampled_df)} rows to: {OUTPUT_PATH}')

Exported 8386 rows to: ..\..\data\machine_learning\bcn_noise_ml_dataset_halved.csv
